In [ ]:
%sql
CREATE OR REPLACE TABLE datacleaning.gold.summary_event_counts
AS
SELECT 
    event_type,                          
    COUNT(event_type) AS total_count    
FROM datacleaning.silver.events
WHERE event_type IN ('purchase', 'cancelled', 'view', 'wishlist', 'cart') 
GROUP BY event_type;

In [ ]:
%sql
CREATE OR REPLACE TABLE datacleaning.gold.fct_sales_performance_summary AS
SELECT 
    p.category AS product_category,
    p.brand AS product_brand,
    COUNT(DISTINCT oi.order_id) AS total_orders,
    SUM(oi.quantity) AS total_units_sold,
    ROUND(SUM(oi.quantity * oi.item_price), 2) AS gross_revenue, -- Fixed column name here
    ROUND(SUM(oi.quantity * oi.item_price) / COUNT(DISTINCT oi.order_id), 2) AS average_order_value -- Fixed column name here
FROM datacleaning.silver.order_items oi
INNER JOIN datacleaning.silver.orders o 
    ON oi.order_id = o.order_id
INNER JOIN datacleaning.silver.products p 
    ON oi.product_id = p.product_id
WHERE o.order_status != 'cancelled'
GROUP BY p.category, p.brand;
